# ResBlock3D

Now that we understand how `CausalConv3D` works, we can start building larger components used inside modern video VAEs.

One of the most important building blocks is the:

```text
ResBlock3D
```

---

Since we are working on a video, we have 2 diffferent things to work on:

- Spatial information
- Temporal information
always seperate these two things and think how the block is processing both of them 

Now let’s first look at the structure of the block so you can better visualize how the input flows through it:

```text
x → GroupNorm → SiLU → CausalConv3D
  → GroupNorm → SiLU → CausalConv3D
  → + skip(x)
```

I’ll explain all the parts one by one, but before that, let’s first understand what this block as a whole is doing and why we need it.

---

Thinking on the Spatial side, if you remember `CausalConv3D` , for the spatial side, it behaves very similarly to a normal convolution layer.
So here, the reblock3d is simple extracting features from the input.
In the early layers, it learns simple things like:

- edges
- textures
- corners

Then as we go deeper into the network, it starts learning more complex structures like:

- shapes
- objects
- scene layouts

This is very similar to how standard CNNs process images.

---

Thinking on the temporal side, In `CausalConv3D`, we usually use a temporal kernel size of `3` or `5`. So what this block is doing is, its looking at those 3 frames at a time and understanding the "physics" of the scene. 
For Example: If you are halfway through a blink in Frame 1 and 2, the 3D kernel helps the model realize that in Frame 3, the eye should be closing further, not suddenly popping wide open.

If you say 3 or 5 frames are very less, we dont have only 1 resblock3d, we have multiple stacked over each other so

```text
Layer 1 sees 3 frames
Layer 2 sees 3 "Layer 1 summaries"
Layer 3 sees even larger summaries
```
By the time you get to the deep layers, the **Receptive Field** has grown to cover 15 or 20 frames.

So long story short >  Looking at 3 frames allows the model to calculate **change**. Once it knows the "change" (the delta), it can ensure that the motion is mathematically continuous rather than a series of disconnected snapshots.


Now other things are there to stablity the gradient flow and it introduce nonlinerality same as done LLMs


---


Q1. Why is it called a “Residual Block”?

Because it uses residual(skip) connection. The skip connection allows information and gradients to flow directly through the network without being heavily modified.

This helps:

- train very deep networks
- reduce vanishing gradient problems
- preserve important low-level information
- improve optimization stability

That is why the structure is called a Residual Block.

Q2. Why do we use Pre-Norm instead of Post-Norm?

We use **Pre-Norm** because it provides a cleaner and more stable gradient flow during backpropagation.



Post-Norm Structure

In Post-Norm, normalization happens **after** the residual addition:

$$
y = \text{Norm}(x + F(x))
$$

This can make gradient propagation harder because the gradient must pass through the normalization layer after the skip connection.

As networks become deeper, training may become unstable.



Pre-Norm Structure

In Pre-Norm, normalization happens **before** the transformation:

$$
y = x + F(\text{Norm}(x))
$$

Why is Pre-Norm better?

The skip connection creates a direct path for gradients:

- gradients flow more smoothly
- training becomes more stable
- deep architectures are easier to optimize
- exploding/vanishing gradient issues are reduced

This is especially important in large transformer and diffusion/video generation models.

---

Q3. Why do we use GroupNorm instead of BatchNorm or LayerNorm in video generation?

BatchNorm depends on **batch statistics** such as batch mean and variance.

However, in video generation:

- videos consume huge GPU memory
- batch sizes are usually very small
- small batches produce unstable statistics

As a result:
- normalization becomes noisy
- training becomes unstable
- model performance degrades

So BatchNorm is not ideal for video generation tasks.


Why not LayerNorm?

LayerNorm normalizes across all channels equally.

While it works well in transformers, it may not preserve spatial/channel structure as effectively for convolution-based video models.


Why GroupNorm works better

GroupNorm divides channels into smaller groups and normalizes within each group.

Advantages of GroupNorm

- independent of batch size
- stable even with batch size = 1
- preserves channel-wise feature structure better than LayerNorm in CNN-style architectures

Because of these properties, GroupNorm is commonly used in:
- diffusion models
- video generation models
- image synthesis networks
- memory-intensive vision architectures

In [ ]:
# 🛠️ Full ResBlock3D Implementation

class ResBlock3D(nn.Module):
    """
    3D residual block for the VAE encoder/decoder.

    Structure:
        x → Norm → SiLU → Conv3D(3×3×3)
          → Norm → SiLU → Conv3D(3×3×3)
          → + skip(x)

    The skip connection uses a 1×1×1 conv if in/out channels differ,
    otherwise it's an identity.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        num_groups: int = 32,
    ):
        super().__init__()

        self.norm1 = get_norm(in_channels, num_groups)
        self.conv1 = CausalConv3d(in_channels, out_channels, kernel_size=3)

        self.norm2 = get_norm(out_channels, num_groups)
        self.conv2 = CausalConv3d(out_channels, out_channels, kernel_size=3)

        self.act = nn.SiLU()

        if in_channels != out_channels:
            self.skip = CausalConv3d_1x1(in_channels, out_channels)
        else:
            self.skip = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        residual = self.skip(x)

        x = self.act(self.norm1(x))
        x = self.conv1(x)

        x = self.act(self.norm2(x))
        x = self.conv2(x)

        return x + residual
